In [ ]:
import uuid, time, sys
from datetime import datetime, timezone
from pyspark.sql import Row, functions as F
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, IntegerType
)

try:
    _nb_path = (
        dbutils.notebook.entry_point.getDbutils()
            .notebook().getContext().notebookPath().get()
    )
    if "/files/" in _nb_path:
        _default_lib = _nb_path.split("/files/")[0] + "/files/libs"
        if not _default_lib.startswith("/Workspace"):
            _default_lib = "/Workspace" + _default_lib
    else:
        _default_lib = ""
except Exception:
    _default_lib = ""

dbutils.widgets.text("shared_lib_path", _default_lib)
shared_lib_path = dbutils.widgets.get("shared_lib_path")
sys.path.insert(0, shared_lib_path)

from pipeline_logging import (
    pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert,
    STATUS_RUNNING, STATUS_SUCCEEDED, STATUS_FAILED, STATUS_NO_FILES,
)
import pipeline_utils as Utils


# ---------------------------------------------------------------------------
# catalog is injected as a job parameter by the bundle (${var.catalog}).
# For manual/standalone runs, derive the catalog from the notebook path so
# the correct target's catalog is used. The map below MUST mirror each target's
# `catalog:` variable in databricks.yml; if you rename a catalog there, update
# it here too.
# Path format: .../bundle/vinoworld/<target>/files/...
# ---------------------------------------------------------------------------
_target_catalog_map = {
    "user":       "dev_vinoworld",
    "dev":        "dev_vinoworld",
    "staging":    "staging_vinoworld",
    "prod":       "vinoworld",
    "azure_prod": "vinoworld",
}

try:
    if "/vinoworld/" in _nb_path and "/files/" in _nb_path:
        _target = _nb_path.split("/vinoworld/")[1].split("/files/")[0]
        _targetcatalog = _target_catalog_map.get(_target, "vinoworld")
    else:
        _targetcatalog = "vinoworld"
except Exception:
    _targetcatalog = "vinoworld"

dbutils.widgets.text("catalog", _targetcatalog)
CATALOG   = dbutils.widgets.get("catalog")
BRONZE    = f"{CATALOG}.bronze"
SILVER    = f"{CATALOG}.silver"
GOLD      = f"{CATALOG}.gold"
AUDIT     = f"{CATALOG}.audit"
RAW_FILES = f"/Volumes/{CATALOG}/datafiles/"

import pipeline_logging
pipeline_logging.configure(AUDIT)


# --------------------------------------------------------------------------
# Get pipeline parameters for pipeline_log table info.
# --------------------------------------------------------------------------

LOCAL_PIPELINE_ID = "11111"    # fallback when run manually outside a job


# Pipeline-level identifiers — shared across all notebooks in a pipeline run.
# Read from parent via widgets; fall back to standalone mode if not provided.
dbutils.widgets.text("pipeline_run_id", "")
value = dbutils.widgets.get("pipeline_run_id")

if not value:
    # Running as a Job task — try to get it from the init task's taskValues
    try:
        value = str(dbutils.jobs.taskValues.get(
            taskKey    = "init_pipeline_log",
            key        = "pipeline_run_id",
            debugValue = LOCAL_PIPELINE_ID
        ))
    except Exception:
        value = str(LOCAL_PIPELINE_ID)

PIPELINE_RUN_ID = value